# S6E9 · 07：新增 XGBoost，与 04 的两种旧模型融合

**现在做什么：**用之前能运行 04/05/06 的 Python 内核，按顺序运行本文件。
**Run All 会启动一套新的 XGBoost 五折训练；没有 False/True 开关。**

保留 04 的 CatBoost 和 LightGBM 预测；不重训它们，不使用 05/06 的新 LightGBM，不自动提交 Kaggle，不覆盖旧文件。
只有第 3 节会进行新的训练；第 1–2 节加载函数、检查依赖和旧结果。

| 已有实验（用户截图；不是 07 实测成绩） | 公榜 |
|---|---:|
| 04：CatBoost / LightGBM = 50/50 | **0.94618** |
| 05：旧模型 = 25/75 | 0.94615 |
| 06：新平滑参数，固定 50/50 | 0.94609 |

**07 的真实比赛效果尚未验证。**混合新模型不保证提分；最终要看相同 OOF 上组合后的实际变化。XGBoost 仍是提升树模型，并不因名字不同就自动具有互补性。

不再把任何极小的正增量都当成提交建议。本轮先约定一个**实用筛选规则**：整体 OOF 至少提升 `0.00005` 且至少 `4/5` 个折改善，才导出待评估候选。
这个门槛只是控制实验提交次数，**不是显著性检验，不是已证实最优门槛，也不保证公榜/私榜更高**。所有结果仍保存供检查。


**本文件验证情况：**已通过语法/Notebook 结构检查，以及 XGBoost 3.1.3 下的小型合成数据五折流程测试（测试时降低训练上限）；包含 ID 对齐、未知类别处理、模型保存/载入、融合、完整折复用和损坏缓存拒绝。没有使用真实竞赛 CSV 完成 07 训练。

## 依赖与运行顺序

新增加的包只有 `xgboost`。其他依赖是原环境已有的 `numpy pandas scipy scikit-learn`（以及 Jupyter 常用的 `packaging`）。
若没有 XGBoost，先在**当前 notebook 单独新建一格**运行：

```python
%pip install "xgboost>=2.0,<4"
```

然后重启内核，再 Run All。本文件**不会自动安装或升级任何包**。已装好且能导入时不必重复安装。
程序在训练前打印 `sys.executable` 和版本，缺包会先停下，不会跑到第几折才报错。

使用 CPU 的 `hist` 方法，不设置 `cuda` 或 `mps`。不要同时在两个内核运行同一实验。
训练中断后，重新运行 07 会复用相同数据/版本/配置/代码下完整保存的折；中断的那一折从头训练。

参考：[XGBoost 安装](https://xgboost.readthedocs.io/en/stable/install.html)、[sklearn 接口与早停](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html)。


## 1. 加载独立函数（不训练）

旧特征与目标编码代码延续 04：行内数字位特征、收入末位与前缀、内层五折目标编码（`smoothing=1`）、训练折内频数和类别映射。
外层五折固定 `random_state=42`，每折验证/测试编码只从外层训练折计算。

新增模型是一套 XGBoost：CPU/hist，深度 4，学习率 0.04，上限 5000 轮，200 轮早停，行/列抽样 0.9，数值分箱上限 1024。
这些是预先选定的实验参数，不是声称已验证最优。使用原生类别支持，并保存 UBJSON 模型；模型文件不等于完整特征编码管线。


In [1]:
from __future__ import annotations
import gc
import hashlib
import importlib.metadata
import json
import sys
import types
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from IPython.display import display

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x

def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}

def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p

def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()

def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds

def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}

def save_json(path: Path, value: dict) -> None:
    """Atomic writes prevent incomplete metadata from being marked complete."""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def package_versions() -> dict:
    return {name: importlib.metadata.version(name)
            for name in ("numpy", "pandas", "scipy", "scikit-learn", "xgboost")}

def check_source_files(bundle: dict) -> None:
    require(input_hashes(bundle["project"]) == bundle["hashes"],
            "原始 CSV 已改变，请停止并核对，不要混用不同版本的数据。")
    require(file_hash(bundle["compare_dir"] / "continuation_summary.json") == bundle["summary_sha256"],
            "04 的比较摘要已改变；请重新加载并核对来源。")

def load_v4(project: Path, compare_dir: Path,
            expected_oof: float | None = 0.94598624) -> dict:
    """Read ONLY the explicitly selected V4 run. Never guess the newest folder."""
    project, compare_dir = Path(project), Path(compare_dir)
    summary_file = compare_dir / "continuation_summary.json"
    if not summary_file.is_file():
        raise FileNotFoundError(f"找不到 {summary_file}。请核对 04 第 5 节打印的比较目录。")
    summary = json.loads(summary_file.read_text(encoding="utf-8"))
    require(summary.get("selected_by_development_oof", {}).get("candidate") == "blend_cat50_lgb50",
            "这个目录不是截图中的 04 的 50/50 最优融合，不能悄悄换成别的运行。")
    train, test, y, folds = load_project(project)
    for frame in (train, test):
        for col in NUMERIC:
            require(col in frame, f"原始数据缺少必要列：{col}")
            require(np.isfinite(frame[col].to_numpy(dtype=float)).all(), f"{col} 包含非有限数值")
        require(np.allclose(frame.Annual_Income_USD, frame.Annual_Income_USD.round()), "收入不是预期整数")
        require(np.allclose(frame.Daily_Commute_km * 10, (frame.Daily_Commute_km * 10).round()), "通勤距离不是预期一位小数")
    hashes = input_hashes(project)
    require(summary.get("input_sha256") == hashes,
            "当前数据哈希与 04 记录不一致。请恢复当时的数据，不要跳过检查。")
    cat_dir = Path(summary["baseline_dir"])
    lgb_dir = Path(summary["lightgbm_dir"])
    fingerprint = lgb_dir / "data_fingerprints.json"
    if not fingerprint.is_file():
        raise FileNotFoundError(f"找不到 {fingerprint}。请保留 04 的原 LightGBM 输出目录。")
    require(json.loads(fingerprint.read_text()) == hashes, "04 的 LightGBM 数据版本不一致。")
    cat = read_run(cat_dir, "catboost", train, test, y, folds)
    lgb4 = read_run(lgb_dir, "lightgbm", train, test, y, folds)
    base = {"oof": 0.5 * cat["oof"] + 0.5 * lgb4["oof"],
            "test": 0.5 * cat["test"] + 0.5 * lgb4["test"]}
    saved_oof = align_csv(compare_dir / "oof_blend_cat50_lgb50.csv", train.id,
                          ["target", "prediction"])
    saved_test = align_csv(compare_dir / "submission_next_candidate.csv", test.id, [TARGET])
    require(np.array_equal(saved_oof.target.to_numpy(), y.to_numpy()), "04 融合 OOF 标签不一致。")
    require(np.allclose(saved_oof.prediction, base["oof"], atol=1e-12, rtol=0),
            "04 的融合 OOF 不等于两个来源模型的 50/50。")
    require(np.allclose(saved_test[TARGET], base["test"], atol=1e-12, rtol=0),
            "04 的候选提交不等于来源模型的 50/50。")
    base["auc"] = float(roc_auc_score(y, base["oof"]))
    require(abs(base["auc"] - summary["selected_by_development_oof"]["oof_auc"]) < 1e-10,
            "04 摘要分数与保存的预测不一致。")
    if expected_oof is not None:
        require(abs(base["auc"] - expected_oof) < 1e-8,
                f"载入的融合 OOF {base['auc']:.10f} 与截图的 {expected_oof:.8f} 不一致。")
    bundle = {"project": project, "compare_dir": compare_dir, "cat_dir": cat_dir,
              "lgb04_dir": lgb_dir, "hashes": hashes, "summary_sha256": file_hash(summary_file),
              "train": train, "test": test, "y": y, "folds": folds,
              "base": base, "models": {"catboost": cat, "lgb04": lgb4}}
    check_source_files(bundle)
    print("04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。")
    print("CatBoost 来源：", cat_dir)
    print("LightGBM 来源：", lgb_dir)
    print(f"当前 04 融合 OOF：{base['auc']:.10f}")
    print("当前已报告公榜：0.94618（来自你的截图，不是本程序新跑出的成绩）")
    return bundle

def submission_frame(bundle: dict, pred: np.ndarray) -> pd.DataFrame:
    test = bundle["test"]
    pred = probabilities(pred, len(test), "Submission")
    frame = pd.DataFrame({"id": test.id.to_numpy(), TARGET: pred})
    sample = bundle["project"] / "data/sample_submission.csv"
    if sample.is_file():
        ids = pd.read_csv(sample, usecols=["id"])
        frame = ids.merge(frame, on="id", how="left", validate="one_to_one", sort=False)
    probabilities(frame[TARGET], len(test), "Ordered submission")
    return frame


FEATURE_SOURCE = 'from __future__ import annotations\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport sys\nimport types\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\nfrom IPython.display import display\n\nTARGET = "Will_Buy_EV"\nNUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",\n           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",\n           "Environmental_Concern_Level"]\n\n\ndef features(frame, model_name):\n    """Row-local features only: no labels or full-data target statistics."""\n    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()\n    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),\n                                 ("Daily_Commute_km", 10, 3),\n                                 ("Charging_Stations_Near_Home", 1, 2),\n                                 ("Charging_Stations_Near_Work", 1, 2)]:\n        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)\n        for digit in range(positions):\n            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)\n    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)\n    x["income_last_2"] = income % 100\n    x["income_last_3"] = income % 1000\n    if model_name == "catboost":\n        for col in NUMERIC:\n            scale = 10 if col == "Daily_Commute_km" else 1\n            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)\n    else:\n        x["income_prefix_100"] = income // 100\n        x["income_prefix_1000"] = income // 1000\n    return x\n\ndef encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):\n    """Inner OOF for training TE; outer-training-only maps for validation/test."""\n    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]\n    labels = np.asarray(train_y)\n    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))\n    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()\n    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]\n    for col in keys:\n        encoded = np.full(len(a), np.nan)\n        for fit_idx, hold_idx in inner:\n            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})\n            stats = subset.groupby("key")["y"].agg(["sum", "count"])\n            prior = labels[fit_idx].mean()\n            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)\n            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()\n        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])\n        prior = labels.mean()\n        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)\n        a[f"{col}_te"] = encoded\n        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()\n        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()\n        assert np.isfinite(encoded).all()\n    # Label-free frequency: exclude a training row\'s own count contribution.\n    counts = a["Annual_Income_USD"].value_counts()\n    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)\n    for frame in (b, c):\n        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)\n    for col in cats:\n        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))\n        for frame in (a, b, c):\n            frame[col] = frame[col].astype(dtype)\n    return a, b, c\n\n\n'

# 07 adds one XGBoost configuration; it never fits either of the old models.
import contextlib
import platform
from packaging.version import Version

XGB_SPEC = {
    "n_estimators": 5000,
    "learning_rate": 0.04,
    "max_depth": 4,
    "min_child_weight": 10.0,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "max_bin": 1024,
    "max_cat_to_onehot": 4,
    "early_stopping_rounds": 200,
}
TE_SMOOTHING = 1.0  # Match 04, NOT the 06 experiment.
REFERENCE = "v4_cat50_lgb04_50"
BLEND_SPECS = {
    REFERENCE: 0.0,
    "v7_cat45_lgb04_45_xgb10": 0.10,
    "v7_cat40_lgb04_40_xgb20": 0.20,
}
# A predeclared practical screening rule, NOT statistical significance or an LB forecast.
REVIEW_RULE = {"min_oof_gain": 0.00005, "min_improved_folds": 4}


def check_xgboost_environment() -> dict:
    try:
        import xgboost as xgb
    except (ImportError, OSError) as error:
        raise RuntimeError(
            "当前内核不能导入 XGBoost。缺包时，在本 notebook 单独一格运行：\n"
            '%pip install "xgboost>=2.0,<4"\n'
            "然后重启内核，从头运行 07。原始错误见下方；若是 libomp 等动态库错误，"
            "请保留完整报错，不要反复重训旧模型。"
        ) from error
    require(Version("2.0") <= Version(xgb.__version__) < Version("4.0"),
            "本代码使用 XGBoost 2.x/3.x 接口。请安装 xgboost>=2.0,<4 后重启内核。")
    versions = package_versions()
    print("当前 Python：", sys.executable)
    print("环境版本：", versions)
    print("XGBoost 使用 CPU + hist；旧 CatBoost/LightGBM 只读取预测，不导入或训练。")
    return versions


def runtime_code_fingerprint(*functions) -> str:
    """Ignore notebook execution filenames, but fingerprint executable logic."""
    def norm(value):
        if isinstance(value, types.CodeType):
            return {"bytecode": value.co_code.hex(), "constants": norm(value.co_consts),
                    "names": value.co_names, "vars": value.co_varnames,
                    "free": value.co_freevars, "cells": value.co_cellvars,
                    "argc": value.co_argcount, "posonly": value.co_posonlyargcount,
                    "kwonly": value.co_kwonlyargcount, "flags": value.co_flags,
                    "exception_table": getattr(value, "co_exceptiontable", b"").hex()}
        if isinstance(value, (tuple, list)):
            return [norm(v) for v in value]
        if isinstance(value, (set, frozenset)):
            return {"set": sorted([norm(v) for v in value], key=lambda v: json.dumps(v, sort_keys=True))}
        if isinstance(value, dict):
            return {str(k): norm(v) for k, v in value.items()}
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)
    payload = [{"name": f.__name__, "code": norm(f.__code__),
                "defaults": norm(f.__defaults__), "kwdefaults": norm(f.__kwdefaults__)} for f in functions]
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


@contextlib.contextmanager
def training_lock(directory: Path):
    """macOS/Linux process lock; automatically released on interruption/exit."""
    import fcntl
    with (directory / ".training.lock").open("a+") as handle:
        try:
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError("另一个内核正在运行同一套 07。请等待它结束，不要并行运行两份。") from error
        try:
            yield
        finally:
            fcntl.flock(handle.fileno(), fcntl.LOCK_UN)


def fold_paths(out: Path, fold: int) -> dict:
    return {"model": out / f"xgboost_fold{fold}.ubj",
            "indices": out / f"xgboost_fold{fold}_train_indices.npy",
            "prediction": out / f"xgboost_fold{fold}_predictions.npz",
            "schema": out / f"xgboost_fold{fold}_schema.json",
            "history": out / f"xgboost_fold{fold}_history.json"}


def verify_fold_cache(out, fold, signature, train, test, ti, vi):
    paths = fold_paths(out, fold)
    marker = json.loads((out / f"fold{fold}_complete.json").read_text())
    require(marker.get("experiment_signature") == signature and marker.get("fold") == fold,
            f"第 {fold} 折完成标记属于不同实验。")
    require(set(marker.get("file_sha256", {})) == {p.name for p in paths.values()},
            f"第 {fold} 折标记缺少必要文件。")
    for filename, checksum in marker["file_sha256"].items():
        path = out / filename
        require(path.is_file() and file_hash(path) == checksum,
                f"第 {fold} 折缓存缺失/被修改：{path}。停止，不混用。")
    require(np.array_equal(np.load(paths["indices"], allow_pickle=False), ti), "缓存训练索引不一致。")
    with np.load(paths["prediction"], allow_pickle=False) as saved:
        require(np.array_equal(saved["valid_indices"], vi), "缓存验证索引不一致。")
        require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), "缓存验证 ID 不一致。")
        require(np.array_equal(saved["test_ids"], test.id), "缓存测试 ID 不一致。")
        vp = probabilities(saved["valid_prediction"], len(vi), "Cached OOF")
        tp = probabilities(saved["test_prediction"], len(test), "Cached test")
    return vp, tp, int(marker["best_trees"])


def train_xgb07(bundle: dict, threads: int = 4, spec: dict | None = None) -> tuple:
    """Train only new XGBoost; reuse completed folds after integrity checks."""
    import xgboost as xgb
    require(threads == -1 or threads >= 1, "THREADS 应为 -1 或正整数。")
    spec = dict(XGB_SPEC if spec is None else spec)
    require(set(spec) == set(XGB_SPEC), "参数键与 07 预设不一致。")
    require(spec["n_estimators"] >= 1 and spec["max_depth"] >= 1, "非法训练参数。")
    check_source_files(bundle)
    versions = check_xgboost_environment()
    experiment = {
        "models": ["xgboost"], "folds": 5, "seed": 42,
        "train_rows": len(bundle["train"]), "test_rows": len(bundle["test"]),
        "spec": spec, "te_smoothing": TE_SMOOTHING, "threads": threads,
        "device": "cpu", "tree_method": "hist", "enable_categorical": True,
        "input_sha256": bundle["hashes"], "package_versions": versions,
        "python_version": sys.version.split()[0], "platform": platform.platform(),
        "v4_summary_sha256": bundle["summary_sha256"],
        "feature_source_sha256": hashlib.sha256(FEATURE_SOURCE.encode()).hexdigest(),
        "code_sha256": runtime_code_fingerprint(features, encode_fold, train_xgb07, verify_fold_cache),
        "revision": "v7-xgboost-one-configuration-1", "inner_seed_rule": "1041 + fold",
        "model_seed_rule": "41 + fold",
    }
    signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()[:16]
    out = bundle["project"] / "outputs" / f"v7_xgboost_{signature}"
    out.mkdir(parents=True, exist_ok=True)
    with training_lock(out):
        config = out / "config.json"
        if config.exists():
            require(json.loads(config.read_text()) == experiment, "缓存配置不同，停止复用。")
        else:
            require(not any(p.name != ".training.lock" for p in out.iterdir()), "未标记目录不为空，停止。")
            save_json(config, experiment)
            save_json(out / "package_versions.json", versions)
            save_json(out / "data_fingerprints.json", bundle["hashes"])
            (out / "feature_code.py").write_text(FEATURE_SOURCE, encoding="utf-8")
        require(file_hash(out / "feature_code.py") == experiment["feature_source_sha256"],
                "已保存特征代码被修改。")
        require(json.loads((out / "data_fingerprints.json").read_text()) == bundle["hashes"],
                "缓存数据版本不一致。")
        require(json.loads((out / "package_versions.json").read_text()) == versions,
                "缓存环境版本不一致。")
        print("07 新 XGBoost 目录：", out, flush=True)
        print("顺序运行五折；完整折校验后复用，中断折从头训练。", flush=True)
        train, test, y, folds = (bundle[k] for k in ("train", "test", "y", "folds"))
        x, xt = features(train, "lightgbm"), features(test, "lightgbm")
        require(x.columns.tolist() == xt.columns.tolist(), "特征列不一致。")
        oof, test_sum = np.full(len(y), np.nan), np.zeros(len(test))
        coverage = np.zeros(len(y), dtype=np.int8)
        records = []
        for fold, (ti, vi) in enumerate(folds, 1):
            paths = fold_paths(out, fold)
            marker = out / f"fold{fold}_complete.json"
            if marker.is_file():
                vp, tp, best = verify_fold_cache(out, fold, signature, train, test, ti, vi)
                print(f"复用 XGBoost Fold {fold}/5；best_trees={best}", flush=True)
            else:
                print(f"\n========== XGBoost | Fold {fold}/5 ==========", flush=True)
                # Same V4 row-local features and INNER cross-fitted TE, smoothing=1.
                # Do NOT fit an encoder to all outer-fold labels before splitting.
                a, b, c = encode_fold(x.iloc[ti], y.iloc[ti], x.iloc[vi], xt,
                                      seed=1041 + fold, smoothing=TE_SMOOTHING)
                require(a.columns.tolist() == b.columns.tolist() == c.columns.tolist(), "编码列不一致。")
                cats = a.select_dtypes(include=["category"]).columns.tolist()
                for col in cats:
                    require(a[col].dtype == b[col].dtype == c[col].dtype, "类别映射不一致。")
                for frame in (a, b, c):
                    require(not frame.select_dtypes(include=["object", "string"]).columns.tolist(),
                            "XGBoost 输入仍包含未转换字符串列。")
                    require(np.isfinite(frame.drop(columns=cats).to_numpy(dtype=float)).all(),
                            "数值/目标编码包含非有限值。")
                model = xgb.XGBClassifier(
                    **spec, objective="binary:logistic", eval_metric="auc",
                    tree_method="hist", device="cpu", enable_categorical=True,
                    random_state=41 + fold, n_jobs=threads, verbosity=1,
                )
                # Early stopping belongs in the constructor (spec), NOT fit().
                model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], verbose=400)
                require(np.array_equal(model.classes_, [0, 1]), "正类概率顺序不正确。")
                best = int(model.best_iteration) + 1
                vp = probabilities(model.predict_proba(b, iteration_range=(0, best))[:, 1], len(vi), "New OOF")
                tp = probabilities(model.predict_proba(c, iteration_range=(0, best))[:, 1], len(test), "New test")
                # UBJSON preserves native categorical splits and best_iteration metadata.
                model.save_model(paths["model"])
                np.save(paths["indices"], ti)
                temporary = paths["prediction"].with_suffix(".npz.tmp")
                with temporary.open("wb") as handle:
                    np.savez_compressed(handle, valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                        valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
                temporary.replace(paths["prediction"])
                save_json(paths["schema"], {
                    "feature_columns": a.columns.tolist(), "dtypes": {col: str(a[col].dtype) for col in a},
                    "categories": {col: a[col].cat.categories.tolist() for col in cats},
                    "best_trees": best, "te_smoothing": TE_SMOOTHING,
                    "note": "Rebuild TE maps from saved outer-training indices, original train.csv and feature_code.py. The model alone is not the entire pipeline.",
                })
                save_json(paths["history"], model.evals_result())
                save_json(marker, {"fold": fold, "experiment_signature": signature, "best_trees": best,
                                   "file_sha256": {p.name: file_hash(p) for p in paths.values()}})
                del a, b, c, model
                gc.collect()
            oof[vi], coverage[vi] = vp, coverage[vi] + 1
            test_sum += tp
            auc = float(roc_auc_score(y.iloc[vi], vp))
            records.append({"model": "xgboost", "fold": fold, "auc": auc, "best_trees": best})
            pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
            print(f"Fold {fold} OOF AUC: {auc:.8f}", flush=True)
        require((coverage == 1).all(), "每行必须恰好有一份折外预测。")
        probabilities(oof, len(y), "Complete XGBoost OOF")
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
            out / "oof_xgboost.csv", index=False)
        # Kept for provenance/read_run; do not submit this single-model file by default.
        pd.DataFrame({"id": test.id.to_numpy(), TARGET: test_sum / len(folds)}).to_csv(
            out / "submission_xgboost.csv", index=False)
        check_source_files(bundle)
        result = read_run(out, "xgboost", train, test, y, folds)
        save_json(out / "summary.json", {"oof_auc": result["auc"], "signature": signature,
                  "note": "Development OOF only; use the separate blend comparison for the next action."})
    print(f"新 XGBoost 五折已完成。单模型 OOF：{result['auc']:.8f}。继续运行第 4 节。")
    return out, result


def choose_candidate(comparison: pd.DataFrame, rule: dict) -> tuple[str, str]:
    """Operational screening; not a significance test."""
    require(rule["min_oof_gain"] > 0 and 1 <= rule["min_improved_folds"] <= 5, "筛选规则非法。")
    rows = comparison.to_dict("records")
    raw_best = max(rows, key=lambda row: row["oof_auc"])
    eligible = [r for r in rows if r["candidate"] != REFERENCE
                and r["delta_vs_v4_blend"] >= rule["min_oof_gain"]
                and r["improved_folds_vs_v4"] >= rule["min_improved_folds"]]
    if eligible:
        return max(eligible, key=lambda row: row["oof_auc"])["candidate"], "REVIEW_CANDIDATE"
    if raw_best["candidate"] != REFERENCE and raw_best["delta_vs_v4_blend"] > 0:
        return REFERENCE, "SMALL_OR_INCONSISTENT_CHANGE"
    return REFERENCE, "KEEP_V4"


def compare_v07(bundle: dict, new: dict, new_dir: Path) -> tuple:
    check_source_files(bundle)
    train, test, y, folds = (bundle[k] for k in ("train", "test", "y", "folds"))
    experiment = json.loads((new_dir / "config.json").read_text())
    signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()[:16]
    require(experiment.get("input_sha256") == bundle["hashes"], "新模型数据来源不一致。")
    require(experiment.get("v4_summary_sha256") == bundle["summary_sha256"], "新模型引用了不同的 04。")
    for fold, (ti, vi) in enumerate(folds, 1):
        verify_fold_cache(new_dir, fold, signature, train, test, ti, vi)
    checked = read_run(new_dir, "xgboost", train, test, y, folds)
    for part in ("oof", "test"):
        require(np.allclose(checked[part], new[part], atol=1e-12, rtol=0), "新预测与磁盘结果不一致。")
    new = checked
    base = bundle["base"]
    predictions, records, per_fold = {}, [], []
    base_folds = [float(roc_auc_score(y.iloc[vi], base["oof"][vi])) for _, vi in folds]
    for name, weight in BLEND_SPECS.items():
        pred = {part: (1 - weight) * base[part] + weight * new[part] for part in ("oof", "test")}
        probabilities(pred["oof"], len(y), name + " OOF")
        probabilities(pred["test"], len(test), name + " Test")
        score = float(roc_auc_score(y, pred["oof"]))
        deltas = []
        for fold, (_, vi) in enumerate(folds, 1):
            auc = float(roc_auc_score(y.iloc[vi], pred["oof"][vi]))
            delta = auc - base_folds[fold - 1]
            deltas.append(delta)
            per_fold.append({"candidate": name, "fold": fold, "auc": auc, "delta_vs_v4_blend": delta})
        records.append({"candidate": name, "xgboost_weight": weight, "oof_auc": score,
                        "delta_vs_v4_blend": score - base["auc"],
                        "improved_folds_vs_v4": sum(d > 0 for d in deltas), "worst_fold_delta_vs_v4": min(deltas)})
        predictions[name] = pred
    comparison = pd.DataFrame(records).sort_values("oof_auc", ascending=False, kind="stable").reset_index(drop=True)
    by_fold = pd.DataFrame(per_fold)
    selected, status = choose_candidate(comparison, REVIEW_RULE)
    raw_best = max(records, key=lambda row: row["oof_auc"])
    models = {"catboost_old": bundle["models"]["catboost"], "lightgbm04_old": bundle["models"]["lgb04"],
              "xgboost07_new": new, "v4_blend_reference": base}
    diagnostic = pd.DataFrame([
        {"model": name, "oof_auc": float(roc_auc_score(y, p["oof"]))} for name, p in models.items()
    ])
    # Correlation is descriptive only. No threshold here is used to select a model.
    pred_table = pd.DataFrame({name: p["oof"] for name, p in models.items()})
    rank_corr = pred_table.corr(method="spearman")
    residual_corr = pred_table.sub(y.to_numpy(), axis=0).corr(method="pearson")
    out = bundle["project"] / "outputs" / ("v7_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    comparison.to_csv(out / "comparison.csv", index=False)
    by_fold.to_csv(out / "comparison_by_fold.csv", index=False)
    diagnostic.to_csv(out / "single_model_diagnostic.csv", index=False)
    rank_corr.to_csv(out / "oof_rank_correlation.csv")
    residual_corr.to_csv(out / "oof_residual_correlation.csv")
    saved = {"train_ids": train.id.to_numpy(), "target": y.to_numpy(), "test_ids": test.id.to_numpy()}
    for name, pred in predictions.items():
        saved[name + "_oof"], saved[name + "_test"] = pred["oof"], pred["test"]
    np.savez_compressed(out / "blend_predictions_for_review.npz", **saved)
    submission_path = None
    if selected != REFERENCE:
        if np.array_equal(predictions[selected]["test"], base["test"]):
            selected, status = REFERENCE, "KEEP_V4"
        else:
            weight = BLEND_SPECS[selected]
            stem = "cat45_lgb45_xgb10" if weight == 0.10 else "cat40_lgb40_xgb20"
            submission_path = out / f"submission_v07_{stem}.csv"
            submission_frame(bundle, predictions[selected]["test"]).to_csv(submission_path, index=False)
    summary = {
        "status": status, "raw_best_by_development_oof": raw_best,
        "selected_after_practical_screen": selected, "screening_rule": REVIEW_RULE,
        "screening_note": "A predeclared practical rule for avoiding tiny/inconsistent changes; NOT a statistical significance test or evidence of private/public improvement.",
        "reference_oof_auc": base["auc"], "candidate_submission": str(submission_path) if submission_path else None,
        "candidate_xgboost_weights": BLEND_SPECS, "candidates": records,
        "v4_compare_dir": str(bundle["compare_dir"]), "xgboost_dir": str(new_dir),
        "input_sha256": bundle["hashes"], "v4_summary_sha256": bundle["summary_sha256"],
        "prior_public_scores_from_user_screenshots": {"v4": 0.94618, "v5_old_models_25_75": 0.94615, "v6": 0.94609},
        "validation_note": "Same five development folds have been used repeatedly for early stopping and selection. Public feedback has also influenced development. NOT an independent audit. No stacking, learned weight search, or public/private labels.",
        "new_kaggle_score": None,
    }
    save_json(out / "summary.json", summary)
    print("【主比较：04 基线 + 两个固定的三模型融合】")
    print(comparison.to_string(index=False, float_format=lambda v: f"{v:.8f}"))
    print("\n结果目录：", out)
    print("本轮 OOF 最高：", raw_best["candidate"])
    print(f"实用筛选规则：OOF 至少 +{REVIEW_RULE['min_oof_gain']:.5f} 且至少 {REVIEW_RULE['min_improved_folds']}/5 折改善；不是显著性检验。")
    if submission_path:
        print("结论：有待评估候选。先看主表和逐折变化，不自动替换 04。")
        print("唯一候选 CSV：", submission_path)
    elif status == "SMALL_OR_INCONSISTENT_CHANGE":
        print("结论：变化过小或逐折不一致。本轮先不提交，保留 04 的 0.94618；没有生成候选 CSV。")
    else:
        print("结论：没有通过本轮筛选的新融合。本轮不提交，保留 04；没有生成候选 CSV。")
    print("公榜、私榜是否提高仍未知；新模型单独 AUC 和相关性都不是替代融合评估的标准。")
    return out, comparison, by_fold, diagnostic, rank_corr, summary


## 2. 检查环境和 04 的来源（不训练）

默认路径来自你已完成的 04 截图。程序从该目录 `continuation_summary.json` 中读取确切的旧 CatBoost / LightGBM 输出目录，而不是寻找“最新目录”。
会核对原始 CSV 哈希、ID、标签、外层五折、旧 OOF 与测试集五折平均。正常情况下打印 04 融合 OOF 约 `0.94598624`。

保持原 `data`、03 的 CatBoost 输出、04 的 LightGBM 输出与 04 比较目录即可；不需要再次运行任何旧 notebook。


In [4]:
versions = check_xgboost_environment()
PROJECT = Path("/Users/yw/unsw/Kaggle")
V4_COMPARE_DIR = PROJECT / "outputs/v4_compare_20260930_163707_027699"
THREADS = 4  # XGBoost 顺序跑五折。更改线程数会产生新的缓存配置。

bundle = load_v4(PROJECT, V4_COMPARE_DIR)
print("检查通过。接下来第 3 节只训练 XGBoost。")

当前 Python： /opt/miniconda3/envs/dl/bin/python
环境版本： {'numpy': '2.4.6', 'pandas': '3.0.6', 'scipy': '1.17.1', 'scikit-learn': '1.9.1', 'xgboost': '3.2.0'}
XGBoost 使用 CPU + hist；旧 CatBoost/LightGBM 只读取预测，不导入或训练。
04 结果核对通过。不会重训旧 CatBoost 或旧 LightGBM。
CatBoost 来源： /Users/yw/unsw/Kaggle/outputs/v3_20260925_180457_278867
LightGBM 来源： /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778
当前 04 融合 OOF：0.9459862434
当前已报告公榜：0.94618（来自你的截图，不是本程序新跑出的成绩）
检查通过。接下来第 3 节只训练 XGBoost。


## 3. 只训练新的 XGBoost 五折

**运行下一格就会开始训练，不需要修改训练开关。**
保存到新 `outputs/v7_xgboost_<配置指纹>/`，每折保存模型、训练索引、验证/测试预测、类别映射、学习曲线及校验标记。
这个目录中的 `submission_xgboost.csv` 是单模型记录，**不是本轮默认提交文件**；请继续第 4 节做融合。


In [5]:
xgb07_dir, xgb07_result = train_xgb07(bundle, threads=THREADS)
print("继续运行第 4 节，检验加入 XGBoost 后是否改善。")

当前 Python： /opt/miniconda3/envs/dl/bin/python
环境版本： {'numpy': '2.4.6', 'pandas': '3.0.6', 'scipy': '1.17.1', 'scikit-learn': '1.9.1', 'xgboost': '3.2.0'}
XGBoost 使用 CPU + hist；旧 CatBoost/LightGBM 只读取预测，不导入或训练。
07 新 XGBoost 目录： /Users/yw/unsw/Kaggle/outputs/v7_xgboost_96e13f4b873f058f
顺序运行五折；完整折校验后复用，中断折从头训练。

========== XGBoost | Fold 1/5 ==========
[0]	validation_0-auc:0.92093
[400]	validation_0-auc:0.94491
[800]	validation_0-auc:0.94505
[1195]	validation_0-auc:0.94505
Fold 1 OOF AUC: 0.94506913

========== XGBoost | Fold 2/5 ==========
[0]	validation_0-auc:0.92293
[400]	validation_0-auc:0.94556
[800]	validation_0-auc:0.94572
[1200]	validation_0-auc:0.94571
[1233]	validation_0-auc:0.94571
Fold 2 OOF AUC: 0.94574462

========== XGBoost | Fold 3/5 ==========
[0]	validation_0-auc:0.92470
[400]	validation_0-auc:0.94676
[800]	validation_0-auc:0.94690
[988]	validation_0-auc:0.94689
Fold 3 OOF AUC: 0.94690607

========== XGBoost | Fold 4/5 ==========
[0]	validation_0-auc:0.92339
[400]	valida

## 4. 比较三行：04 基线与两个固定三模型融合

| 主比较候选 | CatBoost | 04 LightGBM | 新 XGBoost |
|---|---:|---:|---:|
| 原 04 | 50% | 50% | 0% |
| 新融合 A | 45% | 45% | 10% |
| 新融合 B | 40% | 40% | 20% |

即 `新融合 = (1 - XGBoost权重) × 原04融合 + XGBoost权重 × 新XGBoost`。
只测试这两个事先固定的新增权重；没有密集权重搜索，没有 stacking，没有使用任何公榜/私榜标签。

单模型分数和预测相关性单独保存，用于理解新模型；它们**不替代融合 OOF**，相关性低也不等于必然有帮助。
同一 OOF 已经反复用于早停和选择、公榜反馈也影响了实验方向，所以这不是独立测试。


In [7]:
comparison_dir, comparison, by_fold, diagnostic, rank_corr, result_summary = compare_v07(
    bundle, xgb07_result, xgb07_dir
)
print("\n主比较表：")
display(comparison.style.format({
    "xgboost_weight": "{:.0%}", "oof_auc": "{:.8f}",
    "delta_vs_v4_blend": "{:+.8f}", "worst_fold_delta_vs_v4": "{:+.8f}",
}))
print("逐折变化：")
display(by_fold.pivot(index="fold", columns="candidate", values="delta_vs_v4_blend")
        .style.format("{:+.8f}"))
print("单模型诊断（不是提交排名）：")
display(diagnostic.style.format({"oof_auc": "{:.8f}"}))
print("主表截图发来，先不自动提交。状态：", result_summary["status"])

【主比较：04 基线 + 两个固定的三模型融合】
              candidate  xgboost_weight    oof_auc  delta_vs_v4_blend  improved_folds_vs_v4  worst_fold_delta_vs_v4
v7_cat40_lgb04_40_xgb20      0.20000000 0.94603645         0.00005021                     5              0.00004024
v7_cat45_lgb04_45_xgb10      0.10000000 0.94601387         0.00002763                     5              0.00002239
      v4_cat50_lgb04_50      0.00000000 0.94598624         0.00000000                     0              0.00000000

结果目录： /Users/yw/unsw/Kaggle/outputs/v7_compare_20260930_192916_009415
本轮 OOF 最高： v7_cat40_lgb04_40_xgb20
实用筛选规则：OOF 至少 +0.00005 且至少 4/5 折改善；不是显著性检验。
结论：有待评估候选。先看主表和逐折变化，不自动替换 04。
唯一候选 CSV： /Users/yw/unsw/Kaggle/outputs/v7_compare_20260930_192916_009415/submission_v07_cat40_lgb40_xgb20.csv
公榜、私榜是否提高仍未知；新模型单独 AUC 和相关性都不是替代融合评估的标准。

主比较表：


,candidate,xgboost_weight,oof_auc,delta_vs_v4_blend,improved_folds_vs_v4,worst_fold_delta_vs_v4
0,v7_cat40_lgb04_40_xgb20,20%,0.94603645,+0.00005021,5,+0.00004024
1,v7_cat45_lgb04_45_xgb10,10%,0.94601387,+0.00002763,5,+0.00002239
2,v4_cat50_lgb04_50,0%,0.94598624,+0.00000000,0,+0.00000000


逐折变化：


candidate,v4_cat50_lgb04_50,v7_cat40_lgb04_40_xgb20,v7_cat45_lgb04_45_xgb10
fold,,,
1,+0.00000000,+0.00005009,+0.00002754
2,+0.00000000,+0.00005450,+0.00002971
3,+0.00000000,+0.00004024,+0.00002239
4,+0.00000000,+0.00005272,+0.00002874
5,+0.00000000,+0.00005708,+0.00003089


单模型诊断（不是提交排名）：


,model,oof_auc
0,catboost_old,0.94561823
1,lightgbm04_old,0.94594638
2,xgboost07_new,0.94602884
3,v4_blend_reference,0.94598624


主表截图发来，先不自动提交。状态： REVIEW_CANDIDATE


## 看完结果后怎么办？

**先看第 4 节主比较表和逐折变化，不用自己在多个 CSV 中猜。**

若打印 `KEEP_V4` 或 `SMALL_OR_INCONSISTENT_CHANGE`：这一轮先保留 04 的公榜 `0.94618`，不提交；没有导出新候选 CSV。
若打印 `REVIEW_CANDIDATE`：程序只导出一份符合本轮实用筛选规则的候选，并打印完整路径：

- `submission_v07_cat45_lgb45_xgb10.csv`，或
- `submission_v07_cat40_lgb40_xgb20.csv`。

两者只会生成所选的一份，不自动上传，也不会替换旧提交。**即使通过筛选，也还不知道 Kaggle 分数是否提高。**
本地比较目标是 04 OOF 约 `0.94598624`，实际提交后才与公榜 `0.94618` 比较。

保存文件：`comparison.csv`、`comparison_by_fold.csv`、`single_model_diagnostic.csv`、相关性诊断、所有固定融合的预测 `.npz`、含来源和筛选理由的 `summary.json`。
`oof` / `comparison` 文件不能作为比赛提交。

### 技术与来源

旧管线来自本会话的 `04_reuse_catboost_lightgbm.ipynb`、`06_lightgbm_smoothing10.ipynb`。07 为新增 XGBoost 及有限权重概率融合。

参考：[XGBoost 类别支持及模型保存](https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html)、[XGBoost 参数](https://xgboost.readthedocs.io/en/stable/parameter.html)、[概率融合](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier)、[重复模型选择与评估](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)。
